# Setup, version control, and AI literacy

This reference sets up the working environment for everything that follows: moving around a filesystem from the shell, tracking work with git, authenticating to github over ssh, and working safely with ai-generated code. Unlike the lecture subchapters, most commands here run in a terminal rather than in a notebook cell, so they are given as copy-ready snippets for macOS, Linux, and Windows. The few executable cells demonstrate os-independent paths and a silent failure in generated code.

:::{admonition} Learning objectives
:class: tip
- Navigate a filesystem from the shell, and recognise how paths differ across operating systems.
- Build os-independent paths in python with pathlib.
- Track a project with git: init, status, add, commit, branch, push, and pull.
- Authenticate to github over ssh and clone a repository.
- Apply the rule for generated code: pin units and constraints, and never run code you have not read.
:::

## The shell and how paths differ across operating systems

The shell is a text interface for running commands. Three commands cover basic navigation: print the working directory, list its contents, and change directory. The syntax differs between the unix-style shells (zsh on macOS, bash on Linux) and PowerShell on Windows.

```bash
# macOS (zsh) and Linux (bash)
pwd                   # print working directory
ls -la                # list all files, long format
cd ../data            # up one level, then into data/
```

```powershell
# Windows (PowerShell)
Get-Location          # print working directory (alias: pwd)
Get-ChildItem         # list files (aliases: ls, dir)
Set-Location ..\data  # change directory (alias: cd ..\data)
```

The most visible difference is the path separator: unix uses `/`, Windows uses `\`. Hard-coding either makes code non-portable. In python, never build paths by string concatenation; use `pathlib`.

In [1]:
import os
from pathlib import Path, PurePosixPath, PureWindowsPath

print(os.name)        # 'posix' on macOS/Linux, 'nt' on Windows
print(Path.cwd())     # current working directory, in the native format

# build a path without hard-coding a separator
data_file = Path("data") / "raw" / "temperature.csv"
print(data_file)

# the same logical path rendered for each operating system
print(PurePosixPath("data/raw/temperature.csv"))     # forward slashes
print(PureWindowsPath("data/raw/temperature.csv"))   # backslashes

posix
/home/claude
data/raw/temperature.csv
data/raw/temperature.csv
data\raw\temperature.csv


## Version control with git

git records snapshots of a project so you can review history, revert mistakes, and collaborate. The core loop is: edit files, stage the changes you want to keep, then commit them with a message. `git status` shows the current state at every step.

```bash
git init                              # start tracking the current folder
git status                            # what has changed, what is staged
git add analysis.py                   # stage one file (git add . stages all)
git commit -m "Add lapse-rate fit"    # record a snapshot with a message
git log --oneline                     # compact history
```

A commit is a labelled checkpoint. Commit small, coherent changes often; a good message says *why*, not just *what*.

Branches let you develop a change in isolation, then merge it back.

```bash
git switch -c experiment   # create and move onto a new branch (older git: git checkout -b)
# ... edit and commit on 'experiment' ...
git switch main            # return to the main branch
git merge experiment       # bring the experiment's commits into main
```

A *remote* is a copy of the repository hosted elsewhere, usually on github. Push sends your commits up; pull brings others' commits down.

```bash
git remote add origin git@github.com:your-user/your-repo.git
git push -u origin main    # first push sets the upstream; later just: git push
git pull                   # fetch and merge remote changes
```

## Authenticating to github over ssh and cloning

ssh keys let git talk to github without typing a password each time. You generate a keypair once, add the *public* key to your github account, and keep the *private* key secret on your machine.

```bash
# macOS / Linux: generate an ed25519 keypair, then add ~/.ssh/id_ed25519.pub
# to github under Settings -> SSH and GPG keys
ssh-keygen -t ed25519 -C "you@example.com"
ssh -T git@github.com            # test the connection
git clone git@github.com:org/repo.git
```

```powershell
# Windows (PowerShell, built-in OpenSSH): identical commands;
# the key lives in %USERPROFILE%\.ssh\id_ed25519.pub
ssh-keygen -t ed25519 -C "you@example.com"
ssh -T git@github.com
git clone git@github.com:org/repo.git
```

## Working with ai-generated code

ai assistants accelerate coding but shift the burden: the code runs, looks reasonable, and may still be wrong in ways that do not raise an error. The failures that matter most in science are silent and data-dependent — often a units or constraint mismatch the assistant was never told about. Below, a chatbot is asked to flag freezing conditions, and the prompt never pinned the temperature unit.

In [2]:
def is_freezing(temperature):     # unit unspecified: the latent bug
    return temperature < 0

# our station reports temperature in kelvin
print(is_freezing(268.15))   # 268.15 K = -5 °C, should be True
print(is_freezing(271.15))   # 271.15 K = -2 °C, should be True

False
False


:::{admonition} Diagnosis: an unstated unit assumption
:class: warning
Both calls return `False`, yet both temperatures are below freezing. The function compares against `0`, which is correct only for degrees Celsius; in kelvin nothing is ever below zero, so freezing is never detected. The code is not wrong in isolation — it is wrong because the prompt never fixed the unit. The fix makes the unit explicit in the name, the annotation, and the threshold.
:::

In [3]:
def is_freezing_kelvin(temp_kelvin: float) -> bool:
    # threshold expressed in the same unit as the input
    return temp_kelvin < 273.15

print(is_freezing_kelvin(268.15))   # -5 °C -> True
print(is_freezing_kelvin(275.15))   # 2 °C  -> False

True
False


:::{admonition} Computational-thinking fundamental: you own the code you did not write
:class: important
Generated code is a draft you are accountable for. Two habits make it safe: state the constraints in the prompt (units, valid ranges, dtypes, edge cases — "temperature in kelvin; flag values below 273.15 K"), and never run code you have not read line by line. Running unread code risks both wrong science and unsafe actions on your machine.
:::

:::{admonition} Quick exercise: pin the constraint
:class: note
Write the prompt you would give an assistant for a function that converts wind speed from km h⁻¹ to m s⁻¹ and flags gale force, so that units and the threshold leave no ambiguity. (Gale force begins at 17.2 m s⁻¹.)
:::

:::{admonition} Solution
:class: note dropdown
A prompt that pins everything the function needs:

> Write a python function `is_gale_force(wind_kmh: float) -> bool`. The input is wind speed in km h⁻¹. Convert to m s⁻¹ by dividing by 3.6, and return True when the result is at least 17.2 m s⁻¹. Do not round the intermediate value.
:::

:::{admonition} Going deeper: ssh keys
:class: seealso dropdown
- ed25519 is the current recommended key type; rsa (at least 4096 bits) is the fallback for old servers.
- The private key (`id_ed25519`) never leaves your machine; only the public key (`id_ed25519.pub`) goes to github.
- An ssh-agent caches the key so you do not retype the passphrase: run `eval "$(ssh-agent -s)"` then `ssh-add ~/.ssh/id_ed25519`.
- Manage several keys with a `~/.ssh/config` file, one `Host` block per remote.
:::

:::{admonition} Going deeper: .gitignore
:class: seealso dropdown
A `.gitignore` file lists patterns git should not track — keep data, environments, and caches out of history.

```text
# environments and caches
.venv/
__pycache__/
*.pyc

# data and outputs (version the code, archive the data elsewhere)
data/
*.nc
*.zarr/
```

Commit the `.gitignore` itself. Large or binary data does not belong in git; archive it with a doi instead (see the Zenodo box).
:::

:::{admonition} Going deeper: ruff and pre-commit
:class: seealso dropdown
ruff is a fast linter and formatter; pre-commit runs checks automatically before each commit.

```bash
uv tool install ruff
ruff check .          # lint
ruff format .         # format
```

Add a `.pre-commit-config.yaml` with the ruff hooks, then run `pre-commit install` once; from then on a commit is blocked until the staged code passes the checks.
:::

:::{admonition} Going deeper: archiving code and data with a Zenodo doi
:class: seealso dropdown
Zenodo mints a permanent, citable doi for a github release. Link your github account to Zenodo, enable the repository, then publish a tagged release on github; Zenodo archives that snapshot and issues a doi. Cite the doi in papers so others can retrieve the exact version used. This is the standard route to making code and small datasets reproducible and FAIR.
:::

:::{admonition} Takeaways
:class: danger
- Paths differ by os (`/` vs `\`); build them with pathlib, never by string concatenation.
- The git loop is edit → `add` → `commit`; `git status` shows state at each step, and branches isolate work in progress.
- ssh authentication keeps the private key local and uploads only the public key to github.
- Generated code is a draft you are accountable for: pin units and constraints in the prompt, and never run code you have not read.
- Version code in git; archive data and tagged releases with a doi (Zenodo), not in the repository.
:::

## Resources

- [Software Carpentry — The Unix Shell](https://swcarpentry.github.io/shell-novice/) — hands-on introduction to shell navigation and file manipulation.
- [Software Carpentry — Version Control with Git](https://swcarpentry.github.io/git-novice/) — the standard first course on the git workflow used above.
- [Project Pythia — Getting started with GitHub](https://foundations.projectpythia.org/foundations/getting-started-github/) — geoscience-oriented walkthrough of github, ssh setup, cloning, and branches.